# 03 — Model optimisation & training (Step 3)
Train and tune **two models** — a **Neural Network (Keras)** and a **Random Forest (scikit-learn)** — for
each scenario: the **X=0 no-neighbour baseline** plus every `X` × neighbour-encoding (`pos` / `agg`)
combination. Hyperparameters are tuned with **cross-validation** (kept light to respect the ~30-min
Colab budget). We record MSE/MAE/R² and **training duration**.

In [ ]:
# === Project config — Team 8: Throughput Prediction in a Dense 5G deployment (with Transfer Learning) ===
RANDOM_SEED      = 42
RESAMPLE_SECONDS = 120           # time granularity: THE dataset-size knob (professor's guidance: coarsen the
                                 # time grid rather than subsample users). Raw cadence is ~3s with frequent 4s
                                 # steps, duplicate stamps and gaps up to 7s (ACC) / ~1s (Salt&Tar); each output row aggregates every raw sample in a fixed-width window.
N_USERS          = None          # ACC Arena users. None = ALL (~12k), so the X-closest neighbourhoods are the
                                 # real ones; an int (e.g. 1500, seeded random sample) biases the neighbourhoods
                                 # (X closest searched within the sample) and is only for quick debug runs.
N_USERS_SALT     = 3000          # Salt&Tar users: ALL of them (only ~1/3 are ever active; activity is id-biased)
X_VALUES         = [3, 5, 10]    # number of closest users to experiment with. X=1 excluded by design:
                                 # heavy co-location makes a single arbitrary neighbour uninformative.
                                 # X=0 (no neighbour features) IS produced and trained: it is the baseline
                                 # that quantifies what the closest-users features actually add.
ENCODINGS        = ["pos", "agg"]# neighbour-feature encodings: "pos" = ordered per-neighbour columns
                                 # (nb0_*, nb1_*, ...), "agg" = order-invariant aggregates over the X
                                 # neighbours (sum/mean/count) — rationale in notebook 02.
BEST_X           = 3             # X used for the transfer-learning experiment (pick the best from notebook 04)
BEST_ENC         = "pos"         # encoding used for the transfer-learning experiment (pick from notebook 04)
OUTLIER_PCT      = None          # optional train-only sensitivity analysis; primary results keep the full distribution.
                                 # EDA (notebook 01): the ~1% extreme samples carry ~2/3 of the total variance.
ACTIVE_ONLY      = False          # primary task covers every user; True is an optional active-only sensitivity run
MIN_TRAFFIC_TYPE = 2             # keep traffic_type >= this (2=const_rate, 3=video, 4=gaming, 5=http)

# --- data location ---
DRIVE_FOLDER_ID  = "1s1YCWyhN_Fv5sQraTVs4Rga-ATiKPRgo"   # shared Drive folder containing the zip
ZIP_NAME         = "L5GHDD_Dataset.zip"
DATA_ROOT        = "data/raw"     # dataset folders live here after unzip (local default)
PROCESSED_DIR    = "data/processed"
RESULTS_DIR      = "results"

import os, numpy as np, random
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/models", exist_ok=True)

In [ ]:
# === Colab: mount Drive and make outputs PERSIST across notebooks (no-op locally) ===
def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if in_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR   = "/content/drive/MyDrive/5g_throughput_team8"   # persistent project folder on your Drive
    PROCESSED_DIR = f"{PROJECT_DIR}/processed"                     # 02 writes here, 03/04/05 read from here
    RESULTS_DIR   = f"{PROJECT_DIR}/results"                       # models, metrics.csv, figures
    print("Outputs persist in:", PROJECT_DIR)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/models", exist_ok=True)

In [ ]:
# === Evaluation metrics ===
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
def evaluate(y_true, y_pred):
    mse = float(mean_squared_error(y_true, y_pred))
    return {"MSE": mse, "RMSE": float(np.sqrt(mse)),
            "MAE": float(mean_absolute_error(y_true, y_pred)),
            "R2": float(r2_score(y_true, y_pred))}

In [ ]:
import json, time, joblib
import tensorflow as tf
from tensorflow import keras
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, GroupKFold
tf.random.set_seed(RANDOM_SEED)
print("TF:", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

def stem(x, enc):
    return f"acc_X{x}" + ("_agg" if enc == "agg" else "")

def load_xy(x, enc="pos"):
    data = np.load(f"{PROCESSED_DIR}/{stem(x, enc)}.npz")
    return tuple(data[name] for name in ["X_train", "y_train", "groups_train", "X_val", "y_val",
                                              "X_test", "y_test", "traffic_test"])


## Neural network
Small MLP. We do a tiny grid over width/learning-rate using the validation set, then keep the best.

In [ ]:
def build_mlp(input_dim, units=64, lr=1e-3, depth=2):
    model = keras.Sequential([keras.layers.Input((input_dim,))])
    for _ in range(depth):
        model.add(keras.layers.Dense(units, activation="relu"))
    model.add(keras.layers.Dense(1))
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="mse", metrics=["mae"])
    return model

NN_TUNE_MAX_USERS = 300
NN_CANDIDATES = [
    {"units": 64, "depth": 2, "lr": 1e-3},
    {"units": 128, "depth": 2, "lr": 1e-3},
    {"units": 64, "depth": 3, "lr": 3e-4},
]

def train_nn(Xtr, ytr, groups, Xva, yva):
    """Group-aware CV for tuning; final refit uses all train rows and the held-out validation users."""
    all_users = np.unique(groups)
    selected_users = np.random.default_rng(RANDOM_SEED).choice(
        all_users, size=min(NN_TUNE_MAX_USERS, len(all_users)), replace=False)
    tune_mask = np.isin(groups, selected_users)
    Xt, yt, gt = Xtr[tune_mask], ytr[tune_mask], groups[tune_mask]
    folds = list(GroupKFold(3).split(Xt, yt, gt))
    scores = []
    for cfg in NN_CANDIDATES:
        fold_mse = []
        for fold_train, fold_val in folds:
            keras.backend.clear_session()
            keras.utils.set_random_seed(RANDOM_SEED)
            model = build_mlp(Xtr.shape[1], **cfg)
            stop = keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)
            model.fit(Xt[fold_train], yt[fold_train], validation_data=(Xt[fold_val], yt[fold_val]),
                      epochs=20, batch_size=1024, callbacks=[stop], verbose=0)
            fold_mse.append(model.evaluate(Xt[fold_val], yt[fold_val], verbose=0)[0])
        scores.append(float(np.mean(fold_mse)))
    best_cfg = NN_CANDIDATES[int(np.argmin(scores))]
    keras.backend.clear_session()
    keras.utils.set_random_seed(RANDOM_SEED)
    best = build_mlp(Xtr.shape[1], **best_cfg)
    stop = keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
    best.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=40, batch_size=1024,
             callbacks=[stop], verbose=0)
    return best, {**best_cfg, "cv_mse": min(scores)}


## Random forest
`GridSearchCV` (3-fold) over a small grid satisfies the cross-validation requirement.

**Speed (without hurting validity):** the default `max_features` makes RF regression evaluate *all* features
at every split → very slow with 73 features on Colab's 2 vCPUs. We use `max_features="sqrt"` (standard, faster,
usually better), `max_samples=0.3` (each tree fits on a 30% bootstrap subsample — this is just bagging), and
**tune on a capped subsample**, then refit the best config on the full training set. Benchmarked on the real
data (270k train rows): accuracy is essentially unchanged while fitting ~4× faster.

In [ ]:
RF_TUNE_MAX_USERS = 300

def train_rf(Xtr, ytr, groups):
    all_users = np.unique(groups)
    selected_users = np.random.default_rng(RANDOM_SEED).choice(
        all_users, size=min(RF_TUNE_MAX_USERS, len(all_users)), replace=False)
    mask = np.isin(groups, selected_users)
    Xt, yt, gt = Xtr[mask], ytr[mask], groups[mask]
    folds = list(GroupKFold(3).split(Xt, yt, gt))
    base = RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=-1,
                                 max_features="sqrt", min_samples_leaf=2, max_samples=0.3)
    grid = {"n_estimators": [100, 200], "max_depth": [16, None]}
    search = GridSearchCV(base, grid, cv=folds, scoring="neg_mean_squared_error", n_jobs=1)
    search.fit(Xt, yt)
    best = search.best_estimator_
    best.fit(Xtr, ytr)
    return best, {**search.best_params_, "cv_mse": -search.best_score_}


## Train both models for every scenario and save results
Scenarios: `(X=0, none)` baseline + `X ∈ X_VALUES` × `enc ∈ {pos, agg}` — 7 datasets × 2 models.
The baseline tells us how much the closest-users features add at all; pos-vs-agg tells us which
representation of the neighbours works better under heavy co-location.

In [ ]:
def infer_ms(predict, X):
    start = time.time(); predict(X); return round((time.time() - start) / len(X) * 1000, 4)

SCENARIOS = [(0, "none")] + [(x, enc) for x in X_VALUES for enc in ENCODINGS]
results = []
for x, enc in SCENARIOS:
    Xtr, ytr, groups, Xva, yva, Xte, yte, traffic_test = load_xy(x, enc)
    file_stem = stem(x, enc)

    start = time.time(); nn, nn_cfg = train_nn(Xtr, ytr, groups, Xva, yva); nn_s = time.time() - start
    nn_pred = nn.predict(Xte, verbose=0).ravel()
    nn_metrics = evaluate(yte, nn_pred)
    nn_metrics.update(model="NN", X=x, enc=enc, train_s=round(nn_s, 1),
                      infer_ms=infer_ms(lambda values: nn.predict(values, verbose=0), Xte), cfg=str(nn_cfg))
    nn.save(f"{RESULTS_DIR}/models/nn_{file_stem.removeprefix('acc_')}.keras")
    np.savez_compressed(f"{RESULTS_DIR}/models/pred_nn_{file_stem}.npz",
                        y_true=yte, y_pred=nn_pred, traffic_type=traffic_test)

    start = time.time(); rf, rf_cfg = train_rf(Xtr, ytr, groups); rf_s = time.time() - start
    rf_pred = rf.predict(Xte)
    rf_metrics = evaluate(yte, rf_pred)
    rf_metrics.update(model="RF", X=x, enc=enc, train_s=round(rf_s, 1),
                      infer_ms=infer_ms(rf.predict, Xte), cfg=str(rf_cfg))
    joblib.dump(rf, f"{RESULTS_DIR}/models/rf_{file_stem.removeprefix('acc_')}.pkl")
    np.savez_compressed(f"{RESULTS_DIR}/models/pred_rf_{file_stem}.npz",
                        y_true=yte, y_pred=rf_pred, traffic_type=traffic_test)
    results += [nn_metrics, rf_metrics]
    print(f"X={x:>2} {enc:<4} | NN R2={nn_metrics['R2']:.3f} | RF R2={rf_metrics['R2']:.3f}")

import pandas as pd
results = pd.DataFrame(results)
results.to_csv(f"{RESULTS_DIR}/metrics.csv", index=False)
results


Metrics (now including the `enc` column) saved to `results/metrics.csv`; models to `results/models/`
(`*_X0*` = baseline, `*_X{x}*` = positional, `*_X{x}_agg*` = aggregated). Notebook **04** compares them.